In [0]:
import delta

In [0]:
df_full = spark.read.format("parquet").load("/Volumes/raw/upsell/full_load/customers/")
(df_full.coalesce(1)
    .write
    .format("delta")
    .saveAsTable("bronze.upsell.customers"))

In [0]:
(spark.read
    .format("parquet")
    .load("/Volumes/raw/upsell/cdc/customers/")
    .createOrReplaceTempView("customers"))

In [0]:
query = '''
select * from customers
qualify ROW_NUMBER() OVER (partition by idCustomer order by modified_date desc) = 1
'''
df_cdc_unique = spark.sql(query)
df_cdc_unique.display()

In [0]:
bronze = delta.DeltaTable.forName(spark,"bronze.upsell.customers")

In [0]:
#UPSERT 
(bronze.alias("b") 
    .merge(df_cdc_unique.alias("d"), "b.idCustomer = d.idCustomer")
    .whenMatchedDelete(condition = "d.OP = 'DELETE'")
    .whenMatchedUpdateAll(condition = "d.OP ='UPDATE'") 
    .whenNotMatchedInsertAll(condition = "d.OP = 'INSERT' or d.OP = 'UPDATE' ") 
    .execute() 
)

In [0]:
%sql
SELECT * 
FROM bronze.upsell.customers